# Lecture 5 — Neural Networks

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 5.  
**Merged with:** Stanford CS231n 2026 [Lecture 4](https://cs231n.stanford.edu/slides/2026/lecture_4.pdf) (neural networks, first half), the image-features part of [Lecture 5](https://cs231n.stanford.edu/slides/2026/lecture_5.pdf), and the activation functions of [Lecture 6](https://cs231n.stanford.edu/slides/2026/lecture_6.pdf).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture; the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there; **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**; explanations come before it and **"What to notice"** after important results.
* Every function says **what it computes, what goes in, and what comes out**.
* New ideas start with a **tiny example you can check by hand**. **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**The problem:** linear classifiers aren't very powerful (Lecture 3). **Geometrically**, they can only draw straight boundaries; **visually**, they learn only one template per class. Two ways out:
1. **Feature transforms**: turn the image into features where a linear classifier works (hand-designed: color histograms, HOG, bag of words).
2. **Neural networks**: **learn** the feature transform, together with the classifier.

| § | Topic | What you'll build |
|---|---|---|
| 1 | Feature transforms; image features | Color histogram, **HOG**, **bag of visual words** on digits |
| 2 | **Neural networks**: $f = W_2\max(0, W_1x)$ | The slides' ~20-line network; a 2-layer classifier, step by step, gradient-checked |
| 3 | A hidden layer = a bank of templates | Fixing the "two-headed horse" |
| 4 | Why the nonlinearity matters; **activation functions** | Collapse without one; saturation; dead ReLUs; GELU |
| 5 | Network size vs. regularization | Spiral decision boundaries |
| 6 | **Space warping** | A 2-unit hidden layer, visualized |
| 7 | **Universal approximation** | Functions built from ReLU "bumps" |
| 8 | **Convex vs. non-convex** | Testing convexity along random lines |
| 9 | Biological neurons (careful!) | — |
| 10 | PyTorch: an MLP on CIFAR-10 | (Colab) |

### Setup

In [ ]:
import numpy as np                         # arrays and math
import matplotlib.pyplot as plt            # plots
from scipy import ndimage                  # image filters (for HOG features)
from scipy.special import erf              # used by the GELU activation
from sklearn.datasets import load_digits   # the 8x8 handwritten digits
import math, time

np.set_printoptions(precision=4, suppress=True)

**Two datasets:** the **digits** (real images) and the **spiral** (the classic 2-D example from CS231n: three interleaved spiral arms, which no straight line can separate).

In [ ]:
digits = load_digits()
images, labels = digits.images / 16.0, digits.target
split = np.random.default_rng(0).permutation(len(images))
tr, va, te = split[:1000], split[1000:1300], split[1300:]

def make_spiral(n_per_class=100, K=3, noise=0.2, seed=0):
    # K spiral arms of n_per_class points each. Returns (points (N, 2), labels (N,)).
    r = np.random.default_rng(seed); X, y = [], []
    for k in range(K):
        radius = np.linspace(0.0, 1, n_per_class)                                     # from the center outward
        theta = np.linspace(k * 4, (k + 1) * 4, n_per_class) + r.normal(0, noise, n_per_class)   # each arm turns
        X.append(np.c_[radius * np.sin(theta), radius * np.cos(theta)]); y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)

X_sp, y_sp = make_spiral()
plt.figure(figsize=(3.5, 3.5)); plt.scatter(*X_sp.T, c=y_sp, cmap="brg", s=8); plt.title("the spiral dataset"); plt.gca().set_aspect("equal"); plt.show()

**Helpers from Lecture 3:** the softmax loss (as a function of the scores) and a simple linear classifier, as our baseline.

In [ ]:
def softmax_xent(S, y):
    # Softmax cross-entropy loss of scores S (N, C) with labels y. Returns (loss, dL/dS).
    S = S - S.max(1, keepdims=True)
    P = np.exp(S); P /= P.sum(1, keepdims=True)
    loss = -np.log(P[np.arange(len(y)), y]).mean()
    dS = P.copy(); dS[np.arange(len(y)), y] -= 1           # probabilities - one-hot targets
    return loss, dS / len(y)

def train_linear(X, y, n_classes, reg=1e-3, lr=0.5, steps=2000):
    # Linear softmax classifier with a bias, trained by gradient descent. Returns a predict(X) function.
    Xb = np.hstack([X, np.ones((len(X), 1))]); W = np.zeros((Xb.shape[1], n_classes))
    for _ in range(steps):
        _, dS = softmax_xent(Xb @ W, y)
        W -= lr * (Xb.T @ dS + 2 * reg * W)
    return lambda Xq: np.argmax(np.hstack([Xq, np.ones((len(Xq), 1))]) @ W, axis=1)

---
## 1. One solution: feature transforms

Apply a function $\phi$ to the input, then a linear classifier to $\phi(x)$. Lecture 3's example: points arranged in circles can't be split by a line in $(x, y)$, but in **polar coordinates** $(r, \theta)$ they can, and a straight boundary there is a circle back in the original space. **[CS231n 2026]** uses the same example to motivate nonlinearity.

### 1.1 Image features
The classic pre-2012 pipeline: **pixels → hand-designed features → linear classifier**. Three standard features:
* **Color histogram:** ignore where pixels are; count how many fall into each color (here: brightness) bin. Unaffected by rotation or position, but blind to shape.
* **Histogram of oriented gradients (HOG)** (Lowe 1999; Dalal & Triggs 2005): compute the edge direction and strength at each pixel, split the image into small regions, and in each region make a histogram of edge directions weighted by strength. Slides' example: a 320×240 image, 8×8 regions, 9 direction bins → $40\times30\times9 = 10{,}800$ numbers. Captures local shape and tolerates small shifts.
* **Bag of visual words** (data-driven!): (1) cut random small patches from training images and **cluster** them into a "codebook" of typical patches ("visual words"); (2) describe each image by a **histogram of which visual words** its patches look like.

Features are often concatenated. **The key point: the feature extractor is fixed; only the classifier on top is learned.** Neural networks learn the whole pipeline instead.

**Feature 1: an intensity histogram.** Tiny example: a 2×2 image with brightness [0.1, 0.1, 0.6, 0.9] and 4 bins over [0, 1]:

In [ ]:
def color_histogram(ims, bins=8):
    # Fraction of each image's pixels falling into each of `bins` equal brightness ranges over [0, 1].
    return np.stack([np.histogram(im, bins=bins, range=(0, 1))[0] for im in ims]) / ims[0].size

print("histogram of [0.1, 0.1, 0.6, 0.9] with 4 bins:", color_histogram(np.array([[[0.1, 0.1], [0.6, 0.9]]]), bins=4))

(Half the pixels are in the darkest quarter, a quarter in the third, a quarter in the fourth.) **Feature 2: HOG.** For each image: upsample to 16×16, compute the gradient (Lecture 1's Sobel idea) at every pixel, then for each of 4×4 cells, add up gradient strength into 9 direction bins: 4×4×9 = 144 numbers.

In [ ]:
def hog(ims, cell=4, n_bins=9):
    # HOG-style features: a 9-bin gradient-direction histogram (weighted by strength) for each 4x4 cell of a 16x16 image.
    feats = []
    for im in ims:
        up = ndimage.zoom(im, 2, order=1)                                       # 8x8 -> 16x16
        gy, gx = np.gradient(ndimage.gaussian_filter(up, 0.7))
        strength, direction = np.hypot(gx, gy), np.arctan2(gy, gx) % np.pi       # direction in [0, pi) ("unsigned")
        bins = np.minimum((direction / np.pi * n_bins).astype(int), n_bins - 1)
        f = [np.bincount(bins[i:i + cell, j:j + cell].ravel(), weights=strength[i:i + cell, j:j + cell].ravel(), minlength=n_bins)
             for i in range(0, 16, cell) for j in range(0, 16, cell)]
        f = np.concatenate(f)
        feats.append(f / (np.linalg.norm(f) + 1e-9))                            # normalize to length 1
    return np.array(feats)

print("HOG features per image:", hog(images[:1]).shape[1])
print(f"(the slides' 320x240 example: {(320 // 8) * (240 // 8) * 9:,} numbers)")

**Feature 3: bag of visual words.** Three steps:
1. `extract_patches`: every 3×3 patch of each image (36 per image).
2. `kmeans`: cluster training patches into 32 typical patches (the "codebook"). k-means alternates (a) assign each point to its nearest center, (b) move each center to the mean of its points.
3. `bag_of_words`: for each image, find each patch's nearest visual word and count.

In [ ]:
def extract_patches(ims, size=3):
    # All size x size patches of every image, flattened: shape (n_images * patches_per_image, size*size).
    return np.array([im[i:i + size, j:j + size].ravel()
                     for im in ims for i in range(8 - size + 1) for j in range(8 - size + 1)])

def kmeans(X, k, iters=15, seed=0):
    # Cluster rows of X into k groups. Returns the k centers.
    r = np.random.default_rng(seed)
    centers = X[r.choice(len(X), k, replace=False)].copy()
    for _ in range(iters):
        assign = np.argmin(((X[:, None] - centers[None]) ** 2).sum(-1), axis=1)    # (a) nearest center
        for j in range(k):
            if np.any(assign == j):
                centers[j] = X[assign == j].mean(0)                                 # (b) move to the mean
    return centers

codebook = kmeans(extract_patches(images[tr[:300]])[::3], 32)                       # 32 visual words from training patches

def bag_of_words(ims):
    # Histogram (fraction of patches) of nearest visual words, one row per image.
    out = []
    for im in ims:
        P = extract_patches([im])
        words = np.argmin(((P[:, None] - codebook[None]) ** 2).sum(-1), axis=1)
        out.append(np.bincount(words, minlength=len(codebook)) / len(P))
    return np.array(out)

fig, ax = plt.subplots(2, 8, figsize=(8, 2.2))
for i in range(16):
    ax[i // 8, i % 8].imshow(codebook[i].reshape(3, 3), cmap="gray_r", vmin=0, vmax=1); ax[i // 8, i % 8].axis("off")
plt.suptitle("16 of the 32 learned visual words (3x3 patches)", fontsize=8); plt.show()

**Compare the features** with the same linear classifier (each feature standardized first, so gradient descent behaves):

In [ ]:
feature_sets = {
    "raw pixels": lambda ims: ims.reshape(len(ims), -1),
    "intensity histogram": color_histogram,
    "HOG": hog,
    "bag of visual words": bag_of_words,
    "HOG + bag of words": lambda ims: np.hstack([hog(ims), bag_of_words(ims)]),
}
print(f"{'features':22s} {'dimension':>9s} {'validation accuracy':>20s}")
for name, f in feature_sets.items():
    F_tr, F_va = f(images[tr]), f(images[va])
    mu, sd = F_tr.mean(0), F_tr.std(0) + 1e-6                      # standardize with TRAINING statistics
    predict = train_linear((F_tr - mu) / sd, labels[tr], 10)
    print(f"{name:22s} {F_tr.shape[1]:9d} {np.mean(predict((F_va - mu) / sd) == labels[va]):20.1%}")

**What to notice:** the intensity histogram throws away shape and does poorly. HOG, which encodes local edges, is competitive with raw pixels on these clean digits (and, as Lecture 2 showed, much more robust to shifts). Adding bag-of-words to HOG doesn't help here; on harder datasets, combining complementary features usually did. Every feature was **designed by a person**, and for natural images those designs plateaued.

---
## 2. Neural networks

$$\text{(Before) Linear: } f = Wx\qquad\qquad\text{(Now) 2-layer neural network: } f = W_2\max(0, W_1x)$$
with $x\in\mathbb{R}^D$, $W_1\in\mathbb{R}^{H\times D}$, $W_2\in\mathbb{R}^{C\times H}$ (in practice each layer also has a bias). A **3-layer** network: $f = W_3\max(0, W_2\max(0, W_1x))$. For CIFAR-10: $x$ has 3,072 numbers → a **hidden layer** $h$ of 100 → 10 scores.

**Terminology** **[CS231n 2026]**: "neural network" is a broad term; these are more precisely **fully connected networks** or **multi-layer perceptrons (MLPs)**: every unit in one layer connects to every unit in the next. A "2-layer net" = "1-hidden-layer net". **Depth** = number of layers (weight matrices); **width** = number of units per layer.

### 2.1 The slides' ~20-line network
CS231n's complete training code: **define** the network, run the **forward pass**, compute the **gradients**, take **gradient-descent** steps. (Sigmoid hidden units, squared-error loss, random data, exactly as in the slides.)

In [ ]:
N, D_in, H, D_out = 64, 1000, 100, 10
r = np.random.default_rng(0)
x, y = r.normal(size=(N, D_in)), r.normal(size=(N, D_out))          # define the network and the data
w1, w2 = r.normal(size=(D_in, H)), r.normal(size=(H, D_out))
losses = []
for t in range(2000):
    h = 1 / (1 + np.exp(-x.dot(w1)))                                # forward pass: sigmoid hidden layer
    y_pred = h.dot(w2)
    loss = np.square(y_pred - y).sum(); losses.append(loss)
    grad_y_pred = 2.0 * (y_pred - y)                                # analytical gradients (chain rule)
    grad_w2 = h.T.dot(grad_y_pred)
    grad_h = grad_y_pred.dot(w2.T)
    grad_w1 = x.T.dot(grad_h * h * (1 - h))                         # sigmoid's derivative is h (1 - h)
    w1 -= 1e-4 * grad_w1                                            # gradient descent
    w2 -= 1e-4 * grad_w2
print(f"loss: step 0 {losses[0]:,.0f} -> step 2000 {losses[-1]:.3f}")

### 2.2 A two-layer classifier, step by step
For a batch $X$ ($N\times D$), the **forward pass** is:
$$Z_1 = XW_1 + b_1\ \ (\text{pre-activations}),\qquad H = \max(0, Z_1)\ \ (\text{ReLU}),\qquad S = HW_2 + b_2\ \ (\text{scores}),\qquad L = \text{softmax loss}(S) + \lambda(\|W_1\|^2 + \|W_2\|^2)$$
The **backward pass** applies the chain rule from the loss back to each weight (Lecture 6 makes this systematic):
$$\frac{\partial L}{\partial S} = P - Y,\quad \frac{\partial L}{\partial W_2} = H^\top\frac{\partial L}{\partial S},\quad \frac{\partial L}{\partial H} = \frac{\partial L}{\partial S}W_2^\top,\quad \frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial H}\odot\mathbb{1}[Z_1 > 0],\quad \frac{\partial L}{\partial W_1} = X^\top\frac{\partial L}{\partial Z_1}$$
($\odot$ is elementwise multiplication; ReLU passes gradient only where its input was positive.)

**Step 1: the forward pass on a tiny example** (2 inputs, 3 hidden units, 2 classes, one data point), so you can follow every number:

In [ ]:
x_tiny = np.array([[1.0, -2.0]])
W1_tiny = np.array([[0.5, -1.0, 0.2], [0.3, 0.4, -0.6]]); b1_tiny = np.array([0.1, 0.0, 0.0])
W2_tiny = np.array([[1.0, -1.0], [0.5, 0.5], [-0.3, 0.8]]); b2_tiny = np.zeros(2)
Z1 = x_tiny @ W1_tiny + b1_tiny
H_ = np.maximum(Z1, 0)
S_ = H_ @ W2_tiny + b2_tiny
print("Z1 = x W1 + b1      =", Z1)
print("H  = max(0, Z1)     =", H_, " (the negative entry became 0)")
print("S  = H W2 + b2      =", S_)
print("loss (true class 1) =", round(softmax_xent(S_, np.array([1]))[0], 4))

**Step 2: the full model as a class.** `loss(X, y)` runs the forward pass, and if labels are given, the backward pass too. Weights start at He initialization ($\sqrt{2/\text{fan-in}}$; Lecture 10 explains why).

In [ ]:
class TwoLayerNet:
    # f(x) = W2 relu(W1 x + b1) + b2 (activation="relu"), or with no nonlinearity (activation="none").
    def __init__(self, D, H, C, std=None, seed=0, activation="relu"):
        r = np.random.default_rng(seed)
        std1 = std if std is not None else math.sqrt(2 / D)
        std2 = std if std is not None else math.sqrt(2 / H)
        self.p = {"W1": std1 * r.normal(size=(D, H)), "b1": np.zeros(H),
                  "W2": std2 * r.normal(size=(H, C)), "b2": np.zeros(C)}
        self.activation = activation

    def loss(self, X, y=None, reg=0.0):
        # Forward pass; if y is given, also the backward pass. Returns scores, or (loss, gradients dict).
        p = self.p
        Z1 = X @ p["W1"] + p["b1"]                                       # pre-activations
        Hh = np.maximum(Z1, 0) if self.activation == "relu" else Z1      # hidden layer
        S = Hh @ p["W2"] + p["b2"]                                       # scores
        if y is None:
            return S
        loss, dS = softmax_xent(S, y)
        loss += reg * (np.sum(p["W1"] ** 2) + np.sum(p["W2"] ** 2))
        grads = {"W2": Hh.T @ dS + 2 * reg * p["W2"], "b2": dS.sum(0)}   # output layer
        dH = dS @ p["W2"].T                                              # back into the hidden layer
        dZ1 = dH * (Z1 > 0) if self.activation == "relu" else dH         # through the ReLU
        grads["W1"] = X.T @ dZ1 + 2 * reg * p["W1"]; grads["b1"] = dZ1.sum(0)
        return loss, grads

    def train(self, X, y, lr=0.5, reg=1e-3, steps=3000, batch=None, seed=0):
        # Gradient descent (full batch, or minibatches of size `batch`).
        r = np.random.default_rng(seed)
        for _ in range(steps):
            idx = r.integers(0, len(X), batch) if batch else slice(None)
            _, g = self.loss(X[idx], y[idx], reg)
            for k in self.p:
                self.p[k] -= lr * g[k]
        return self

    def predict(self, X):
        return np.argmax(self.loss(X), axis=1)

**Step 3: gradient check** (Lecture 4): compare each analytic gradient with a numerical estimate at a few random entries.

In [ ]:
def grad_check(net, X, y, reg, h=1e-5, n=8, seed=0):
    # Largest relative error between analytic and numerical gradients over n random entries of each parameter.
    r = np.random.default_rng(seed); _, g = net.loss(X, y, reg); worst = 0
    for k, P in net.p.items():
        for _ in range(n):
            ix = tuple(r.integers(0, s) for s in P.shape); old = P[ix]
            P[ix] = old + h; lp = net.loss(X, y, reg)[0]
            P[ix] = old - h; lm = net.loss(X, y, reg)[0]
            P[ix] = old
            num = (lp - lm) / (2 * h)
            worst = max(worst, abs(num - g[k][ix]) / max(abs(num), abs(g[k][ix]), 1e-10))
    return worst

print(f"two-layer net gradient check: max relative error {grad_check(TwoLayerNet(2, 20, 3, seed=1), X_sp[::10], y_sp[::10], 1e-2):.1e}")

**Step 4: train on the spiral**, and compare with a linear classifier:

In [ ]:
linear_predict = train_linear(X_sp, y_sp, 3)
net = TwoLayerNet(2, 100, 3).train(X_sp, y_sp, lr=1.0, reg=1e-3, steps=5000)
print(f"spiral training accuracy: linear classifier {np.mean(linear_predict(X_sp) == y_sp):.1%},  "
      f"2-layer net (100 hidden units) {np.mean(net.predict(X_sp) == y_sp):.1%}")

**Step 5: draw the decision regions** (classify every point of a grid covering the plot):

In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.2, 1.2, 250), np.linspace(-1.2, 1.2, 250))
G = np.c_[gx.ravel(), gy.ravel()]
region_colors = ["#c6dbef", "#fcbba1", "#c7e9c0"]; point_colors = np.array(["#2171b5", "#cb181d", "#238b45"])

def plot_regions(ax, predict, X, y, title):
    # Color each grid point by its predicted class, then overlay the data.
    ax.contourf(gx, gy, predict(G).reshape(gx.shape), levels=[-0.5, 0.5, 1.5, 2.5], colors=region_colors)
    ax.scatter(*X.T, c=point_colors[y], s=6); ax.set_title(title, fontsize=9); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
plot_regions(ax[0], linear_predict, X_sp, y_sp, "linear classifier")
plot_regions(ax[1], net.predict, X_sp, y_sp, "2-layer neural network")
plt.show()

**What to notice:** the linear classifier can only cut the plane with straight lines; the network bends its boundaries to follow the spiral arms.

---
## 3. What does the hidden layer learn? A bank of templates

* **Linear classifier:** one template per class.
* **2-layer network:** the first layer $W_1$ is a **bank of templates** (100 of them for CIFAR-10 in the slides instead of 10), and the second layer **recombines** them into class scores. Templates can be **shared between classes**, and one class can use **several templates**: one per mode (a horse facing left *and* one facing right).
* The catch: these are **distributed representations**; most individual templates aren't interpretable.

**Lecture 3's experiment, revisited:** training on digits **plus their mirror images** forced each linear template into a symmetric blur and cost accuracy. Does a hidden layer fix it?

**Step 1: data.** (We subtract 0.3, roughly the average pixel, so inputs are centered.)

In [ ]:
Xtr_pix = images[tr].reshape(len(tr), -1) - 0.3
Xva_pix = images[va].reshape(len(va), -1) - 0.3
Xtr_mirror = np.vstack([Xtr_pix, images[tr][:, :, ::-1].reshape(len(tr), -1) - 0.3])   # originals + mirror images
ytr_mirror = np.r_[labels[tr], labels[tr]]
Xva_mirror = images[va][:, :, ::-1].reshape(len(va), -1) - 0.3                         # mirrored validation digits
print("training with mirror images:", Xtr_mirror.shape)

**Step 2: train a linear classifier and two networks, and test on normal and mirrored digits.**

In [ ]:
print(f"{'model (trained on digits + mirror images)':45s} {'normal val':>11s} {'mirrored val':>13s}")
lin_m = train_linear(Xtr_mirror, ytr_mirror, 10)
print(f"{'linear classifier (one template per class)':45s} {np.mean(lin_m(Xva_pix) == labels[va]):11.1%} {np.mean(lin_m(Xva_mirror) == labels[va]):13.1%}")
for H_units in [20, 100]:
    net_m = TwoLayerNet(64, H_units, 10).train(Xtr_mirror, ytr_mirror, lr=0.3, reg=1e-3, steps=3000, batch=200)
    print(f"{f'2-layer net, {H_units} hidden units':45s} {np.mean(net_m.predict(Xva_pix) == labels[va]):11.1%} "
          f"{np.mean(net_m.predict(Xva_mirror) == labels[va]):13.1%}")

**Step 3: look at some first-layer templates** (columns of $W_1$, reshaped to 8×8):

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(14, 3))
for i in range(20):
    ax[i // 10, i % 10].imshow(net_m.p["W1"][:, i].reshape(8, 8), cmap="RdBu_r"); ax[i // 10, i % 10].axis("off")
plt.suptitle("20 of the 100 first-layer templates: many are NOT symmetric, so each mode can get its own", fontsize=9); plt.show()

**What to notice:** the networks handle "this shape OR its mirror image" far better than one template per class: the hidden layer devotes different templates to each orientation, and the second layer combines them.

---
## 4. Why the nonlinearity matters, and activation functions

**Q: What if we build a network without the activation function?**
$$f = W_2W_1x = W_3x\quad\text{with } W_3 = W_2W_1$$
**We end up with a linear classifier again!** (With biases too: $W_2(W_1x + b_1) + b_2$ is still a linear function plus a constant.) Stacking linear layers adds parameters but no power. Let's verify:

In [ ]:
net_lin = TwoLayerNet(2, 100, 3, activation="none").train(X_sp, y_sp, lr=0.1, reg=1e-3, steps=5000)
W3 = net_lin.p["W1"] @ net_lin.p["W2"]                         # the single equivalent matrix
b3 = net_lin.p["b1"] @ net_lin.p["W2"] + net_lin.p["b2"]
print("network without activation == one linear layer (W3 = W1 W2):", np.allclose(net_lin.loss(X_sp), X_sp @ W3 + b3))
print(f"spiral accuracy: no activation {np.mean(net_lin.predict(X_sp) == y_sp):.1%},  linear classifier {np.mean(linear_predict(X_sp) == y_sp):.1%},  "
      f"with ReLU {np.mean(net.predict(X_sp) == y_sp):.1%}")

### 4.1 The activation-function zoo
| Function | Formula | Pros | Cons |
|---|---|---|---|
| **Sigmoid** | $\sigma(x) = \frac{1}{1 + e^{-x}}$ | Output in (0, 1); historically a neuron's "firing rate" | **Saturates**: large $\lvert x\rvert$ kills the gradient; not zero-centered |
| **tanh** | $\tanh(x) = 2\sigma(2x) - 1$ | Zero-centered | Still saturates |
| **ReLU** | $\max(0, x)$ | No saturation for $x > 0$; very cheap; **converges much faster than sigmoid in practice (e.g., 6×)** | Not zero-centered; **dead ReLUs** for $x < 0$ |
| **Leaky ReLU** | $\max(0.01x, x)$ | Never dies | An extra hyperparameter (or learned, PReLU) |
| **ELU** | $x$ if $x > 0$, else $e^x - 1$ | Smooth; closer to zero-mean outputs | Needs $\exp$ |
| **GELU** **[CS231n 2026]** | $x\,\Phi(x)$ ($\Phi$ = standard normal CDF) | Smooth near 0, which **helps training in practice**; **the default in Transformers** | More compute; large negative inputs still have gradient → 0 |
| 🔎 **SiLU / Swish** | $x\,\sigma(x)$ | Similar to GELU; used in EfficientNet and many LLMs | |

**ReLU is a good default choice for most problems.** **Q: Where are activations used in CNNs?** Generally **after linear operators** (fully connected or convolution layers) **[CS231n 2026]**.

**Each function, with its derivative** (the derivative is what backprop multiplies by, so it controls how gradients flow):

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))
activations = {                                       # name: (function, derivative)
    "sigmoid":    (sigmoid, lambda x: sigmoid(x) * (1 - sigmoid(x))),
    "tanh":       (np.tanh, lambda x: 1 - np.tanh(x) ** 2),
    "ReLU":       (lambda x: np.maximum(0, x), lambda x: (x > 0).astype(float)),
    "Leaky ReLU": (lambda x: np.where(x > 0, x, 0.1 * x), lambda x: np.where(x > 0, 1.0, 0.1)),
    "ELU":        (lambda x: np.where(x > 0, x, np.exp(np.minimum(x, 0)) - 1), lambda x: np.where(x > 0, 1.0, np.exp(np.minimum(x, 0)))),
    "GELU":       (lambda x: x * 0.5 * (1 + erf(x / np.sqrt(2))), None),      # derivative computed numerically below
    "SiLU":       (lambda x: x * sigmoid(x), None),
}
xs = np.linspace(-4, 4, 801)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (f, df) in activations.items():
    ax[0].plot(xs, f(xs), label=name)
    ax[1].plot(xs, df(xs) if df is not None else np.gradient(f(xs), xs), label=name)
ax[0].set_title("activation functions"); ax[1].set_title("their derivatives"); ax[0].legend(fontsize=7); ax[0].set_ylim(-1.5, 4)
plt.show()

**Saturation in numbers:** the sigmoid's derivative is at most 0.25 (at 0) and nearly 0 for large inputs. Backprop through many layers multiplies these factors together:

In [ ]:
d_sigmoid = activations["sigmoid"][1]
print(f"sigmoid derivative: at 0 {d_sigmoid(0.0):.2f}, at 5 {d_sigmoid(5.0):.4f}, at 10 {d_sigmoid(10.0):.1e}")
print(f"through 10 sigmoid layers, even in the BEST case the gradient shrinks by 0.25^10 = {0.25 ** 10:.1e}")

**Dead ReLUs, demonstrated.** If a ReLU unit's input is negative for **every** training example, its output and its gradient are zero for every example, so its incoming weights never change: it never recovers. We push half the hidden units' biases far negative and train:

In [ ]:
def dead_fraction(net_, X):
    # Fraction of hidden units whose pre-activation is <= 0 for every input in X.
    return np.mean(np.all(X @ net_.p["W1"] + net_.p["b1"] <= 0, axis=0))

net_dead = TwoLayerNet(2, 100, 3, seed=2)
net_dead.p["b1"][:50] = -10.0                          # push half the units deep into the negative region
print(f"dead before training: {dead_fraction(net_dead, X_sp):.0%}")
net_dead.train(X_sp, y_sp, lr=1.0, reg=1e-3, steps=3000)
print(f"dead after training:  {dead_fraction(net_dead, X_sp):.0%}  (their data gradient is exactly zero, so they stay dead)")

---
## 5. How big should the network be?

**More neurons = more capacity**: a network with more hidden units can draw more complex boundaries (and can fit noise). **Do not use the size of the network as a regularizer. Use stronger regularization instead** **[CS231n 2026, UMich]**. *Demos:* ConvNetJS (cs.stanford.edu/people/karpathy/convnetjs/demo/classify2d.html), TensorFlow Playground (playground.tensorflow.org).

**Experiment:** a **noisier** spiral (easier to overfit), with a separate test spiral. Top row: vary the hidden size at fixed λ. Bottom row: fix a large network (300 units) and vary λ.

In [ ]:
X_noisy, y_noisy = make_spiral(noise=0.45, seed=3)
X_test_sp, y_test_sp = make_spiral(noise=0.45, seed=4)
fig, ax = plt.subplots(2, 4, figsize=(15, 7.5))
for j, H_units in enumerate([3, 10, 50, 300]):
    m = TwoLayerNet(2, H_units, 3, seed=1).train(X_noisy, y_noisy, lr=1.0, reg=1e-4, steps=6000)
    tr_acc, te_acc = np.mean(m.predict(X_noisy) == y_noisy), np.mean(m.predict(X_test_sp) == y_test_sp)
    plot_regions(ax[0, j], m.predict, X_noisy, y_noisy, f"{H_units} hidden, lambda 1e-4\ntrain {tr_acc:.0%}, test {te_acc:.0%}")
    print(f"{H_units:3d} hidden units, lambda 1e-4  : train {tr_acc:.1%}, test {te_acc:.1%}")
for j, lam in enumerate([1e-4, 1e-3, 1e-2, 1e-1]):
    m = TwoLayerNet(2, 300, 3, seed=1).train(X_noisy, y_noisy, lr=1.0, reg=lam, steps=6000)
    tr_acc, te_acc = np.mean(m.predict(X_noisy) == y_noisy), np.mean(m.predict(X_test_sp) == y_test_sp)
    plot_regions(ax[1, j], m.predict, X_noisy, y_noisy, f"300 hidden, lambda {lam:g}\ntrain {tr_acc:.0%}, test {te_acc:.0%}")
    print(f"300 hidden units, lambda {lam:<7g}: train {tr_acc:.1%}, test {te_acc:.1%}")
plt.tight_layout(); plt.show()

**What to notice:**
* **Top row:** more units → more complex boundaries. Here the larger networks also **generalize best** (test accuracy keeps rising with size), so shrinking the network would only have hurt.
* **Bottom row:** the same 300-unit network goes from detailed to smooth to useless as λ grows; λ ≥ 0.01 underfits badly (chance, 33%, at λ = 0.1).
* **λ is a hyperparameter to tune on validation data**: too little risks overfitting (not much of an issue on this data), too much destroys the fit.

---
## 6. Space warping

A linear layer $h = Wx$ only rotates, stretches, and shears space: points that aren't linearly separable stay that way. A **ReLU layer** $h = \max(0, Wx + b)$ **folds** space: every region where a unit's input is negative gets flattened onto that unit's zero. In the folded space the classes can become linearly separable, and the network's final linear layer draws a straight line *there*, which is a bent boundary in the original space.

**Experiment:** classify "one quadrant vs. the rest" (not linearly separable) with only **2 hidden units**, so the hidden space is 2-D and we can plot it. **Step 1: train** (keep the best of 10 random starts):

In [ ]:
warp_rng = np.random.default_rng(5)
Xw = warp_rng.uniform(-1, 1, (400, 2))
yw = ((Xw[:, 0] > 0) & (Xw[:, 1] > 0)).astype(int)                   # the top-right quadrant vs. everything else
best = None
for seed in range(10):
    m = TwoLayerNet(2, 2, 2, seed=seed).train(Xw, yw, lr=1.0, reg=0.0, steps=4000)
    acc = np.mean(m.predict(Xw) == yw)
    if best is None or acc > best[0]:
        best = (acc, m)
acc, m = best
print(f"2 hidden units: training accuracy {acc:.1%}")

**Step 2: look at the data in three spaces**: the input; after the linear part $W_1x + b_1$; after the ReLU. And the final layer's boundary in the ReLU space.

In [ ]:
Z_lin = Xw @ m.p["W1"] + m.p["b1"]                 # after the linear part
H_relu = np.maximum(Z_lin, 0)                      # after the ReLU (folded)
w_diff = m.p["W2"][:, 1] - m.p["W2"][:, 0]; b_diff = m.p["b2"][1] - m.p["b2"][0]   # class 1 wins where w_diff . h + b_diff > 0

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].scatter(*Xw.T, c=yw, cmap="coolwarm", s=8); ax[0].set_title("input space")
ax[1].scatter(*Z_lin.T, c=yw, cmap="coolwarm", s=8); ax[1].axhline(0, c="k", lw=0.5); ax[1].axvline(0, c="k", lw=0.5)
ax[1].set_title("after W1 x + b1 (just a linear map)")
ax[2].scatter(*H_relu.T, c=yw, cmap="coolwarm", s=8); ax[2].set_title("after ReLU (negative parts folded to 0)")
hx = np.linspace(H_relu[:, 0].min(), H_relu[:, 0].max() + 0.1, 2)
if abs(w_diff[1]) > 1e-9:
    ax[2].plot(hx, -(w_diff[0] * hx + b_diff) / w_diff[1], "k--", label="final linear boundary"); ax[2].legend(fontsize=8)
ax[2].set_xlim(H_relu[:, 0].min() - 0.1, H_relu[:, 0].max() + 0.1); ax[2].set_ylim(H_relu[:, 1].min() - 0.1, H_relu[:, 1].max() + 0.1)
plt.tight_layout(); plt.show()
print(f"a straight line in ReLU space separates the classes: {np.mean(((H_relu @ w_diff + b_diff) > 0) == yw):.1%}")

**What to notice:** no single line isolates the quadrant in the input space, but after folding, one straight line in hidden space does. More hidden units → more folds → more complex shapes.

---
## 7. Universal approximation

**A network with one hidden layer can approximate any continuous function $f:\mathbb{R}^N\to\mathbb{R}^M$ (on a bounded region) as closely as you like, given enough hidden units** (Cybenko 1989; Hornik 1991; Leshno et al. 1993 for ReLU).

**The constructive intuition (from the UMich lecture):** a single ReLU $\max(0, x - s)$ is a **hinge** at position $s$. Four hinges, combined with weights $(+1, -1, -1, +1)/\text{width}$, make a **bump**: 0, ramp up, flat at 1, ramp down, 0. **Add up bumps** of different positions and heights, and you can draw any function: a two-layer network with 4 hidden units per bump.

**Step 1: one bump from 4 ReLUs.**

In [ ]:
relu = lambda z: np.maximum(0, z)
def bump(x, s1, s2, s3, s4):
    # A trapezoid: 0 before s1, rising to 1 at s2, flat until s3, falling to 0 at s4. Built from 4 ReLU hinges.
    return relu(x - s1) / (s2 - s1) - relu(x - s2) / (s2 - s1) - relu(x - s3) / (s4 - s3) + relu(x - s4) / (s4 - s3)

x_q = np.linspace(0, 1, 1000)
print("bump(0.3, 0.4, 0.6, 0.7) at x = 0.2, 0.35, 0.5, 0.65, 0.8:", np.round(bump(np.array([0.2, 0.35, 0.5, 0.65, 0.8]), 0.3, 0.4, 0.6, 0.7), 6) + 0.0)   # round away tiny float noise

**Step 2: approximate a curve with a sum of narrow bumps** (one per interval, with height = the target's value at the interval's center):

In [ ]:
target = lambda x: np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x) + x

def sum_of_bumps(x, n_bumps):
    # Approximate `target` on [0, 1] with n_bumps nearly rectangular bumps (4 ReLUs each).
    edges = np.linspace(0, 1, n_bumps + 1); half = (edges[1] - edges[0]) / 2; eps = 1e-3 * half
    out = np.zeros_like(x)
    for i in range(n_bumps):
        c = (edges[i] + edges[i + 1]) / 2
        out += target(c) * bump(x, c - half - eps, c - half + eps, c + half - eps, c + half + eps)
    return out

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(x_q, bump(x_q, 0.3, 0.4, 0.6, 0.7)); ax[0].set_title("one bump = 4 ReLUs")
for a, n_b in [(ax[1], 5), (ax[2], 40)]:
    approx = sum_of_bumps(x_q, n_b)
    err = np.max(np.abs(approx - target(x_q))[(x_q > 0.01) & (x_q < 0.99)])
    a.plot(x_q, target(x_q), "k", lw=1, label="target"); a.plot(x_q, approx, "r", lw=1, label=f"{4 * n_b} ReLUs")
    a.set_title(f"{n_b} bumps: max error {err:.3f}"); a.legend(fontsize=7)
    print(f"{n_b:3d} bumps ({4 * n_b:3d} hidden ReLUs): max approximation error {err:.3f}")
plt.show()

**What to notice:** more bumps → smaller error, without limit. **But be careful** (UMich): universal approximation says a network **can represent** any function; it does **not** say we can **learn** it with SGD, or how much data we need. (**Reminder: kNN is also a universal approximator**, Lecture 2.) The construction can also need exponentially many units in high dimensions; **deep** networks represent many functions far more compactly (Telgarsky 2016).

---
## 8. Convex functions, and why networks aren't

A function is **convex** if for any two points $x_1, x_2$ and any $t\in[0, 1]$:
$$f(tx_1 + (1 - t)x_2)\leq t\,f(x_1) + (1 - t)\,f(x_2)$$
In words: **the graph lies below every chord** (straight line between two points on it). $x^2$ is convex; $\cos x$ isn't. Intuition: **a convex function is a bowl**, and convex functions are **easy to optimize**, with guarantees of reaching the **global minimum**.
* **Linear classifiers optimize a convex function** (Lecture 4 §13).
* **Neural network losses are usually non-convex.** Few guarantees, yet **empirically it seems to work anyway**; an active research area.

**Test convexity directly:** pick random pairs of weight vectors and a random $t$, and check whether the chord inequality is violated (the left side bigger than the right). **Step 1: the test.**

In [ ]:
def convexity_violations(loss_fn, w0, scale, n_lines=200, seed=0):
    # Check f(t a + (1-t) b) <= t f(a) + (1-t) f(b) on random chords around w0.
    # Returns (number of violations, largest violation).
    r = np.random.default_rng(seed); worst, count = 0.0, 0
    for _ in range(n_lines):
        a = w0 + scale * r.normal(size=w0.shape); b_ = w0 + scale * r.normal(size=w0.shape)
        t = r.uniform(0.1, 0.9)
        gap = loss_fn(t * a + (1 - t) * b_) - (t * loss_fn(a) + (1 - t) * loss_fn(b_))   # > 0 means a violation
        count += gap > 1e-10; worst = max(worst, gap)
    return count, worst

**Step 2: both models' losses as functions of one flat weight vector** (on a third of the spiral):

In [ ]:
Xs, ys = X_sp[::3], y_sp[::3]
Xs_b = np.hstack([Xs, np.ones((len(Xs), 1))])
linear_loss = lambda w: softmax_xent(Xs_b @ w.reshape(3, 3), ys)[0]          # 9 weights

shapes = {"W1": (2, 20), "b1": (20,), "W2": (20, 3), "b2": (3,)}
sizes = [int(np.prod(s)) for s in shapes.values()]
def network_loss(w):
    # Unpack a flat vector into the 2-layer net's parameters and return the loss.
    parts = np.split(w, np.cumsum(sizes)[:-1])
    P = {k: v.reshape(s) for (k, s), v in zip(shapes.items(), parts)}
    return softmax_xent(np.maximum(Xs @ P["W1"] + P["b1"], 0) @ P["W2"] + P["b2"], ys)[0]

for name, f_, dim in [("linear softmax classifier", linear_loss, 9), ("2-layer ReLU network", network_loss, sum(sizes))]:
    count, worst = convexity_violations(f_, np.zeros(dim), 2.0)
    print(f"{name:26s}: chord inequality violated on {count}/200 random chords (worst violation {worst:.3f})")

**Step 3: 1-D slices** of each loss along random directions:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
slice_rng = np.random.default_rng(1); ts = np.linspace(-3, 3, 300)
for a, (name, f_, dim) in zip(ax, [("linear softmax: 1-D slices", linear_loss, 9), ("2-layer network: 1-D slices", network_loss, sum(sizes))]):
    w0 = slice_rng.normal(size=dim)
    for _ in range(4):
        d = slice_rng.normal(size=dim)
        a.plot(ts, [f_(w0 + t * d) for t in ts])
    a.set_title(name); a.set_xlabel("step along a random direction")
plt.show()

**What to notice:** the linear loss never violates the inequality (every slice is a bowl). The network's does, on some chords (one counterexample is enough to prove non-convexity), and its slices have bumps and plateaus. Yet gradient descent trained it to near-perfect spiral accuracy in §2. Why non-convex networks train so well is only partly understood (overparameterization, saddles being more common than bad minima, the implicit bias of SGD).

---
## 9. Biological neurons: an analogy to handle with care

A biological **neuron**: **dendrites** carry impulses toward the **cell body**, which sums them; if the total passes a threshold, it fires a spike along its **axon** to **presynaptic terminals**, which connect through **synapses** to other neurons. An artificial neuron mimics this: inputs $x_i$ weighted by "synaptic strengths" $w_i$, summed as $\sum_iw_ix_i + b$, passed through an activation (historically the sigmoid, a "firing rate").

**Be very careful with brain analogies!** Biological neurons come in **many types**; **dendrites can perform complex nonlinear computations**; **synapses are not a single weight but complex nonlinear dynamical systems** (London & Häusser 2005). Brains have **complex connectivity**, while networks are **organized into regular layers for computational efficiency**, though **networks with random connections can work too** (Xie et al. 2019, *randomly wired networks*).

---
## 10. PyTorch: an MLP on CIFAR-10 (run in Colab)

A 2-layer MLP on CIFAR-10 raw pixels reaches roughly **50–55%** test accuracy (vs. ~40% linear, ~35–39% kNN), still far below CNNs (Lecture 7), because it ignores the spatial structure of images.

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads CIFAR-10, ~170MB). Use a GPU runtime.
import torch, torch.nn as nn, torchvision
device = "cuda" if torch.cuda.is_available() else "cpu"
tr_ds = torchvision.datasets.CIFAR10("./data", train=True, download=True); te_ds = torchvision.datasets.CIFAR10("./data", train=False, download=True)
Xtr_t = torch.tensor(tr_ds.data, dtype=torch.float32, device=device).reshape(50000, -1) / 255; ytr_t = torch.tensor(tr_ds.targets, device=device)
Xte_t = torch.tensor(te_ds.data, dtype=torch.float32, device=device).reshape(10000, -1) / 255; yte_t = torch.tensor(te_ds.targets, device=device)
mu, sd = Xtr_t.mean(0), Xtr_t.std(0); Xtr_t, Xte_t = (Xtr_t - mu) / sd, (Xte_t - mu) / sd     # standardize each pixel
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(3072, 512), nn.ReLU(), nn.Linear(512, 10)).to(device)       # a 2-layer network
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-2)
for step in range(5000):
    idx = torch.randint(0, 50000, (256,), device=device)
    loss = nn.functional.cross_entropy(model(Xtr_t[idx]), ytr_t[idx])
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    print(f"MLP test accuracy: {(model(Xte_t).argmax(1) == yte_t).float().mean().item():.1%}")

---
## 11. Summary

| Topic | Takeaway |
|---|---|
| Feature transforms | Linear classifier on $\phi(x)$; hand-designed features (histograms, HOG, bag of words) plateaued |
| Neural networks | $f = W_2\max(0, W_1x)$; fully connected networks / MLPs; depth = layers, width = units |
| Hidden layer | A bank of shared templates; handles multiple modes per class |
| Nonlinearity | Without it the network collapses to a linear classifier |
| Activations | ReLU: good default; sigmoid/tanh saturate; dead ReLUs; GELU/SiLU in modern models |
| Network size | Bigger networks + tuned regularization, not smaller networks |
| Space warping | ReLU layers fold space so classes become linearly separable |
| Universal approximation | Sums of ReLU bumps approximate any continuous function; says nothing about learnability |
| Convexity | Linear classifiers: convex (guarantees); networks: non-convex, yet SGD works |
| Brains | Loosely inspired; be careful with the analogy |

### Check yourself
<details><summary>1. What happens if you remove the ReLU from a 2-layer network?</summary>

W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂): a single linear function. It's a linear classifier with redundant parameters.
</details>

<details><summary>2. Why do sigmoid networks suffer from vanishing gradients?</summary>

The sigmoid's derivative is at most 0.25 and nearly 0 when saturated; backprop multiplies these factors layer after layer, shrinking the gradient exponentially.
</details>

<details><summary>3. What's a dead ReLU, and why doesn't it recover?</summary>

A unit whose input is negative for every example: its output and gradient are zero everywhere, so gradient descent never changes its incoming weights.
</details>

<details><summary>4. One hidden layer is enough to approximate any function. Why use deep networks?</summary>

The theorem guarantees existence, not efficiency or learnability. A shallow network may need exponentially many units, and the theorem says nothing about whether SGD finds the weights or how much data is needed. Depth is far more compact for many functions.
</details>

<details><summary>5. Should you use a smaller network to avoid overfitting?</summary>

Generally no: use a large enough network and control overfitting with regularization (L2, dropout, augmentation). Larger networks are also easier to optimize.
</details>

### Exercises
1. Add a **third layer** to `TwoLayerNet` (derive the gradients, then gradient-check). Does it fit the spiral with fewer total parameters?
2. Replace ReLU with **sigmoid** and count the steps to reach 95% on the spiral (the slides claim ReLU is ~6× faster).
3. Build a **2-D bump** from ReLUs (combine bumps along x and y, then threshold with another ReLU).
4. Use **Leaky ReLU** in §4.1's dead-unit experiment. What happens to the "dead" units?
5. Try bag-of-words codebooks of 8, 32, and 128 words. How does accuracy change?

### References
* CS231n course notes, *Neural Networks Part 1* (cs231n.github.io/neural-networks-1).
* Dalal & Triggs 2005 (HOG); Csurka et al. 2004 (bag of visual words); Lin et al. 2011 (ImageNet 2011 winner).
* Krizhevsky et al. 2012 (ReLU); Maas et al. 2013 (Leaky ReLU); Clevert et al. 2016 (ELU); Hendrycks & Gimpel 2016 (GELU); Ramachandran et al. 2017 (Swish).
* Cybenko 1989; Hornik 1991; Leshno et al. 1993; Telgarsky 2016.
* London & Häusser 2005; Xie et al. 2019.

**Next:** Lecture 6, Backpropagation.